# OC — Evidencia BI vs SQL (Validación técnica)

Este notebook genera evidencia reproducible para auditar coherencia numérica entre KPIs del dashboard **OC** (Power BI) y resultados SQL ejecutados sobre **PostgreSQL (DW_SEM)**.

## Outputs generados
- `outputs/oc_kpis_sql.csv`
- `outputs/oc_kpis_sql.log`
- `outputs/oc_kpis_bi.csv` *(plantilla si no existe)*
- `outputs/oc_kpis_bi_vs_sql.csv`

## Requisitos
- Contenedor Postgres levantado (chile-pg)
- Acceso a DB `chilecompra` con usuario `chile_user`
- Vista: `dw_sem.v_fact_ordenes_compra_sem_v2`


In [1]:

# 0) Imports y parámetros
import os
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine, text

# --- Configuración DB (ajusta si aplica) ---
PG_HOST = os.getenv("PG_HOST", "localhost")
PG_PORT = int(os.getenv("PG_PORT", "5433"))  # tu compose expone 5433->5432
PG_DB   = os.getenv("PG_DB",   "chilecompra")
PG_USER = os.getenv("PG_USER", "chile_user")
PG_PASS = os.getenv("PG_PASS", "CHANGE_ME")  # si usas password, setéalo por env var

# --- Output dir ---
OUT_DIR = os.getenv("OUT_DIR", "./outputs")
os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR =", os.path.abspath(OUT_DIR))

# --- Monedas esperadas (puedes ajustar) ---
MONEDAS_A_EVALUAR = ["CLP", "USD", "EUR", "CLF", "UTM"]

# Importante: en OC el slicer está usando el campo 'moneda' (según tus capturas).
# Si quieres auditar por 'moneda_norm', cambia la variable:
MONEDA_FIELD = os.getenv("MONEDA_FIELD", "moneda")   # "moneda" o "moneda_norm"

# Flags DQ mínimos para moneda (si existen en la vista)
APLICAR_DQ_MONEDA = True

NOW = datetime.now().strftime("%Y-%m-%d %H:%M:%S")


OUT_DIR = /home/engineer/Documents/Proyectos/TFM/docs/evidence/fase7_auditoria_BI/Evidencia_BI_vs_SQL/OC/outputs


In [2]:

# 1) Conexión a PostgreSQL
def make_engine():
    if PG_PASS:
        url = f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}"
    else:
        url = f"postgresql+psycopg2://{PG_USER}@{PG_HOST}:{PG_PORT}/{PG_DB}"
    return create_engine(url)

engine = make_engine()

def fetch_df(sql: str) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

# Smoke test
fetch_df("SELECT 1 AS ok")


,ok
0,1


In [3]:

# 2) Introspección de columnas (evita errores por nombres inexistentes)
VIEW_FULL = "dw_sem.v_fact_ordenes_compra_sem_v2"

sql_cols = '''
SELECT column_name
FROM information_schema.columns
WHERE table_schema = 'dw_sem' AND table_name = 'v_fact_ordenes_compra_sem_v2'
ORDER BY column_name;
'''
df_cols = fetch_df(sql_cols)
cols = set(df_cols["column_name"].tolist())
print("Columnas detectadas:", len(cols))
df_cols.head(20)


Columnas detectadas: 25


,column_name
0,cantidad_items
1,cantidad_total
2,estado_oc
3,fecha_aceptacion_sk
4,fecha_creacion_sk
5,flag_fuente_preferente_missing
6,flag_moneda_missing
7,flag_moneda_unknown
8,flag_organismo_valido
9,flag_sector_missing


In [4]:

# 3) Resolver columnas clave OC (BK/SK/Monto) dinámicamente
def pick_first(candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

COL_OC_BK = pick_first(["orden_compra_bk", "oc_bk", "orden_bk"])
COL_OC_SK = pick_first(["orden_compra_sk", "oc_sk", "orden_sk"])
COL_MONEDA = "moneda" if "moneda" in cols else None
COL_MONEDA_NORM = "moneda_norm" if "moneda_norm" in cols else None
COL_MONTO = pick_first(["monto_total", "monto_oc", "monto", "monto_total_oc"])

COL_PROV_SK = pick_first(["proveedor_sk"])
COL_PROD_ONU_SK = pick_first(["producto_onu_sk"])
COL_ORG_SK = pick_first(["organismo_sk"])

# Flags opcionales
FLAG_MONEDA_MISSING = "flag_moneda_missing" if "flag_moneda_missing" in cols else None
FLAG_MONEDA_UNKNOWN = "flag_moneda_unknown" if "flag_moneda_unknown" in cols else None

print("COL_OC_BK:", COL_OC_BK)
print("COL_OC_SK:", COL_OC_SK)
print("COL_MONEDA:", COL_MONEDA, "| COL_MONEDA_NORM:", COL_MONEDA_NORM, "| MONEDA_FIELD:", MONEDA_FIELD)
print("COL_MONTO:", COL_MONTO)
print("COL_PROV_SK:", COL_PROV_SK, "| COL_PROD_ONU_SK:", COL_PROD_ONU_SK, "| COL_ORG_SK:", COL_ORG_SK)
print("FLAGS:", FLAG_MONEDA_MISSING, FLAG_MONEDA_UNKNOWN)

assert COL_OC_BK or COL_OC_SK, "No se encontró columna BK/SK para OC. Revisa la vista."
assert (MONEDA_FIELD in ["moneda", "moneda_norm"]), "MONEDA_FIELD debe ser 'moneda' o 'moneda_norm'"
assert (MONEDA_FIELD == "moneda" and COL_MONEDA) or (MONEDA_FIELD == "moneda_norm" and COL_MONEDA_NORM),        "El campo elegido en MONEDA_FIELD no existe en la vista."


COL_OC_BK: orden_compra_bk
COL_OC_SK: orden_compra_sk
COL_MONEDA: moneda | COL_MONEDA_NORM: moneda_norm | MONEDA_FIELD: moneda
COL_MONTO: monto_total
COL_PROV_SK: proveedor_sk | COL_PROD_ONU_SK: producto_onu_sk | COL_ORG_SK: organismo_sk
FLAGS: flag_moneda_missing flag_moneda_unknown


In [5]:

# 4) Diagnóstico (moneda vs moneda_norm) — robusto (NO debe romper el notebook)
diag_select = []
diag_select.append(f"{COL_MONEDA or 'NULL'} AS moneda")
diag_select.append(f"{COL_MONEDA_NORM or 'NULL'} AS moneda_norm")
diag_select.append("COUNT(*) AS n_rows")

if COL_OC_BK:
    diag_select.append(f"COUNT(DISTINCT {COL_OC_BK}) AS n_oc_bk")
elif COL_OC_SK:
    diag_select.append(f"COUNT(DISTINCT {COL_OC_SK}) AS n_oc_sk")

if COL_MONTO:
    diag_select.append(f"COALESCE(SUM({COL_MONTO}),0) AS sum_monto_total")

sql_diag = f'''
SELECT
  {", ".join(diag_select)}
FROM {VIEW_FULL}
GROUP BY 1,2
ORDER BY n_rows DESC;
'''

try:
    df_diag = fetch_df(sql_diag)
    df_diag
except Exception as e:
    print("⚠️ Diagnóstico falló (no bloquea el notebook):", repr(e))
    df_diag = None


## 5) Generación de KPIs SQL

Se generan KPIs por moneda (sin globales), alineados al diseño del dashboard OC.

**Monto:** se calcula como suma de `MAX(monto_total)` por OC (SK/BK), para evitar doble conteo cuando la vista contiene múltiples filas por OC.


In [6]:

# 5.1 Helpers SQL
def dq_where_clause():
    clauses = []
    if APLICAR_DQ_MONEDA:
        if FLAG_MONEDA_MISSING:
            clauses.append(f"{FLAG_MONEDA_MISSING} = 0")
        if FLAG_MONEDA_UNKNOWN:
            clauses.append(f"{FLAG_MONEDA_UNKNOWN} = 0")
    return (" AND " + " AND ".join(clauses)) if clauses else ""

def where_moneda(moneda_val: str):
    field = COL_MONEDA if MONEDA_FIELD == "moneda" else COL_MONEDA_NORM
    return f"WHERE {field} = '{moneda_val}'"

def count_distinct(col: str, where_sql: str = ""):
    return f"SELECT COUNT(DISTINCT {col})::bigint AS valor FROM {VIEW_FULL} {where_sql};"

def sum_monto_unico(where_sql: str = ""):
    oc_key = COL_OC_SK or COL_OC_BK
    if not COL_MONTO:
        return None
    return f'''
    SELECT COALESCE(SUM(mx),0)::numeric(38,2) AS valor
    FROM (
      SELECT {oc_key} AS oc_key, MAX({COL_MONTO}) AS mx
      FROM {VIEW_FULL}
      {where_sql}
      {dq_where_clause()}
      GROUP BY 1
    ) t;
    '''

def kpi_row(kpi, moneda_val, valor):
    return {"kpi": kpi, "moneda_val": moneda_val, "valor_sql": float(valor) if valor is not None else None}


In [7]:

# 5.2 Ejecutar KPIs SQL y generar CSV + LOG
log_lines = []
log_lines.append(f"OC KPIs SQL EVIDENCE - {NOW}")
log_lines.append("-"*60)

rows = []

# Globales (NO se incluyen en comparación principal)
# Nota: el dashboard OC no tiene tarjetas globales; solo KPIs por moneda.
oc_key_for_count = COL_OC_BK or COL_OC_SK


# Por moneda
for m in MONEDAS_A_EVALUAR:
    where = where_moneda(m)

    # Conteo
    v = fetch_df(count_distinct(oc_key_for_count, where)).iloc[0,0]
    rows.append(kpi_row("OC_Conteo", m, v))

    # Monto
    q_m = sum_monto_unico(where)
    if q_m:
        v_m = fetch_df(q_m).iloc[0,0]
        rows.append(kpi_row("OC_Monto_Total", m, v_m))

    # Proveedores
    if COL_PROV_SK:
        v = fetch_df(count_distinct(COL_PROV_SK, where)).iloc[0,0]
        rows.append(kpi_row("OC_Proveedores_Distintos", m, v))

    # Productos ONU
    if COL_PROD_ONU_SK:
        v = fetch_df(count_distinct(COL_PROD_ONU_SK, where)).iloc[0,0]
        rows.append(kpi_row("OC_Productos_ONU_Distintos", m, v))

    # Organismos
    if COL_ORG_SK:
        v = fetch_df(count_distinct(COL_ORG_SK, where)).iloc[0,0]
        rows.append(kpi_row("OC_Organismos_Distintos", m, v))

df_sql = pd.DataFrame(rows)

sql_csv_path = os.path.join(OUT_DIR, "oc_kpis_sql.csv")
df_sql.to_csv(sql_csv_path, index=False)

log_path = os.path.join(OUT_DIR, "oc_kpis_sql.log")
with open(log_path, "w", encoding="utf-8") as f:
    f.write("\n".join(log_lines) + "\n\n")
    f.write("## Vista\n")
    f.write(f"{VIEW_FULL}\n\n")
    f.write("## MONEDA_FIELD\n")
    f.write(f"{MONEDA_FIELD}\n\n")
    if df_diag is not None:
        f.write("## Diagnóstico moneda vs moneda_norm\n")
        f.write(df_diag.to_string(index=False))
        f.write("\n\n")
    f.write("## KPIs SQL (tabla larga)\n")
    f.write(df_sql.to_string(index=False))
    f.write("\n")

print("✅ Generados:", sql_csv_path, "y", log_path)
df_sql.head(20)


✅ Generados: ./outputs/oc_kpis_sql.csv y ./outputs/oc_kpis_sql.log


,kpi,moneda_val,valor_sql
0,OC_Conteo,CLP,4.531314e+06
1,OC_Monto_Total,CLP,2.765186e+13
2,OC_Proveedores_Distintos,CLP,3.755000e+04
3,OC_Productos_ONU_Distintos,CLP,9.716000e+03
4,OC_Organismos_Distintos,CLP,8.450000e+02
5,OC_Conteo,USD,1.274400e+04
6,OC_Monto_Total,USD,9.562489e+08
7,OC_Proveedores_Distintos,USD,6.200000e+02
8,OC_Productos_ONU_Distintos,USD,5.250000e+02
9,OC_Organismos_Distintos,USD,6.070000e+02


## 6) Cargar valores BI (manual) y comparar

1) Completa `outputs/oc_kpis_bi.csv` con los valores capturados desde tarjetas (por moneda)  
2) Ejecuta esta sección para obtener `oc_kpis_bi_vs_sql.csv`.


In [8]:

# 6.1 Crear plantilla BI si no existe
bi_csv_path = os.path.join(OUT_DIR, "oc_kpis_bi.csv")

if not os.path.exists(bi_csv_path):
    # Plantilla BI: 5 KPIs x monedas (sin globales)
    kpis = [
        "OC_Conteo",
        "OC_Monto_Total",
        "OC_Proveedores_Distintos",
        "OC_Productos_ONU_Distintos",
        "OC_Organismos_Distintos",
    ]
    rows_tpl = []
    for m in MONEDAS_A_EVALUAR:
        for k in kpis:
            rows_tpl.append({"kpi": k, "moneda_val": m, "valor_pbi": None})
    df_tpl = pd.DataFrame(rows_tpl)
    df_tpl.to_csv(bi_csv_path, index=False)
    print("⚠️ No existía oc_kpis_bi.csv. Plantilla creada en:", bi_csv_path)
else:
    print("✅ Encontrado:", bi_csv_path)

df_bi = pd.read_csv(bi_csv_path)
df_bi.head(20)


✅ Encontrado: ./outputs/oc_kpis_bi.csv


,kpi,moneda_val,valor_pbi
0,OC_Conteo,CLP,4.531314e+06
1,OC_Monto_Total,CLP,2.765186e+13
2,OC_Proveedores_Distintos,CLP,3.755000e+04
3,OC_Productos_ONU_Distintos,CLP,9.716000e+03
4,OC_Organismos_Distintos,CLP,8.450000e+02
5,OC_Conteo,CLF,1.190100e+04
6,OC_Monto_Total,CLF,5.730699e+07
7,OC_Proveedores_Distintos,CLF,1.114000e+03
8,OC_Productos_ONU_Distintos,CLF,6.880000e+02
9,OC_Organismos_Distintos,CLF,5.730000e+02


In [ ]:

# 6.2 Comparación BI vs SQL (robusta, maneja SQL=0)
df_sql_cmp = df_sql.copy()
df_bi_cmp = df_bi.copy()

df_sql_cmp["moneda_val"] = df_sql_cmp["moneda_val"].where(df_sql_cmp["moneda_val"].notna(), None)
df_bi_cmp["moneda_val"]  = df_bi_cmp["moneda_val"].where(df_bi_cmp["moneda_val"].notna(), None)

df_cmp = df_sql_cmp.merge(df_bi_cmp, on=["kpi", "moneda_val"], how="left")

df_cmp["diff_abs"] = (df_cmp["valor_pbi"] - df_cmp["valor_sql"]).abs()

# --- Tolerancias ---
TOL_MONTO_ABS = 1.0  # tolerancia absoluta para montos (precision float)
is_monto = df_cmp["kpi"].astype(str).str.contains("MONTO", case=False, na=False)

# diff_abs efectivo para estado (monto tolera hasta 1)
df_cmp["diff_abs_eff"] = df_cmp["diff_abs"]
df_cmp.loc[is_monto & (df_cmp["diff_abs_eff"] <= TOL_MONTO_ABS), "diff_abs_eff"] = 0.0


df_cmp["diff_pct"] = None
mask_sql_nonzero = df_cmp["valor_sql"].notna() & (df_cmp["valor_sql"] != 0)
df_cmp.loc[mask_sql_nonzero, "diff_pct"] = (df_cmp.loc[mask_sql_nonzero, "diff_abs"] / df_cmp.loc[mask_sql_nonzero, "valor_sql"]) * 100

df_cmp["estado"] = "OK"
df_cmp.loc[df_cmp["valor_pbi"].isna(), "estado"] = "FALTA_PBI"
df_cmp.loc[df_cmp["valor_sql"].isna(), "estado"] = "FALTA_SQL"



mask_no_ok = (df_cmp["estado"] == "OK") & (df_cmp["diff_abs_eff"] != 0)
df_cmp.loc[mask_no_ok, "estado"] = "NO_OK"


mask_sql0 = (df_cmp["estado"] == "OK") & (df_cmp["valor_sql"] == 0) & (df_cmp["valor_pbi"] != 0)
df_cmp.loc[mask_sql0, "estado"] = "NO_OK_SQL0"

cmp_path = os.path.join(OUT_DIR, "oc_kpis_bi_vs_sql.csv")
df_cmp.to_csv(cmp_path, index=False)

print("✅ Comparación generada:", cmp_path)
df_cmp


✅ Comparación generada: ./outputs/oc_kpis_bi_vs_sql.csv


/tmp/ipykernel_363055/1203070121.py:23: FutureWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df_cmp.loc[mask_sql_nonzero, "diff_pct"] = (df_cmp.loc[mask_sql_nonzero, "diff_abs"] / df_cmp.loc[mask_sql_nonzero, "valor_sql"]) * 100


,kpi,moneda_val,valor_sql,valor_pbi,diff_abs,diff_abs_eff,diff_pct,estado
0,OC_Conteo,CLP,4.531314e+06,4.531314e+06,0.000000,0.0,0.000000e+00,OK
1,OC_Monto_Total,CLP,2.765186e+13,2.765186e+13,0.019531,0.0,7.063268e-14,OK
2,OC_Proveedores_Distintos,CLP,3.755000e+04,3.755000e+04,0.000000,0.0,0.000000e+00,OK
3,OC_Productos_ONU_Distintos,CLP,9.716000e+03,9.716000e+03,0.000000,0.0,0.000000e+00,OK
4,OC_Organismos_Distintos,CLP,8.450000e+02,8.450000e+02,0.000000,0.0,0.000000e+00,OK
5,OC_Conteo,USD,1.274400e+04,1.274400e+04,0.000000,0.0,0.000000e+00,OK
6,OC_Monto_Total,USD,9.562489e+08,9.562489e+08,0.000000,0.0,0.000000e+00,OK
7,OC_Proveedores_Distintos,USD,6.200000e+02,6.200000e+02,0.000000,0.0,0.000000e+00,OK
8,OC_Productos_ONU_Distintos,USD,5.250000e+02,5.250000e+02,0.000000,0.0,0.000000e+00,OK
9,OC_Organismos_Distintos,USD,6.070000e+02,6.070000e+02,0.000000,0.0,0.000000e+00,OK
